# 面试题：怎样定位第一处因果错误？

首错不是最后一条错误，而是最早违反前置条件并生成错误 artifact 的事件。要沿 parent/artifact 依赖回溯，用权威状态验证反事实：修复该点下游是否恢复。

## 真实案例

退款 trace 包含 Router 选错、参数错、工具失败、虚假回答和完整分支六个事件。

## 基线

基线把最后一条坏回答当根因。

## 结果解读

手写依赖回溯输出首个异常与传播链。

## 失败案例

修正最后回答不会修复错误工具选择。

In [1]:
events = [{'id':'E1','parent':None,'ok':True,'name':'解析目标'}, {'id':'E2','parent':'E1','ok':False,'name':'Router选错退款工具'}, {'id':'E3','parent':'E2','ok':False,'name':'参数继承错误'}, {'id':'E4','parent':'E3','ok':False,'name':'工具拒绝'}, {'id':'E5','parent':'E4','ok':False,'name':'回复已完成'}, {'id':'E6','parent':'E1','ok':True,'name':'权威订单回读'}]  # 构造六个带父事件的因果 trace 节点。
print('因果事件:', events)  # 输出节点、父边和执行是否正确。
print('教学说明：parent/artifact 边来自结构化 trace，而不是事后由模型猜测。')  # 说明因果证据。

因果事件: [{'id': 'E1', 'parent': None, 'ok': True, 'name': '解析目标'}, {'id': 'E2', 'parent': 'E1', 'ok': False, 'name': 'Router选错退款工具'}, {'id': 'E3', 'parent': 'E2', 'ok': False, 'name': '参数继承错误'}, {'id': 'E4', 'parent': 'E3', 'ok': False, 'name': '工具拒绝'}, {'id': 'E5', 'parent': 'E4', 'ok': False, 'name': '回复已完成'}, {'id': 'E6', 'parent': 'E1', 'ok': True, 'name': '权威订单回读'}]
教学说明：parent/artifact 边来自结构化 trace，而不是事后由模型猜测。


In [2]:
last_bad = [row['id'] for row in events if not row['ok']][-1]  # 构造把最后异常当根因的错误基线。
print('最后错误基线:', last_bad)  # 输出 E5 这个传播末端。
print('基线风险：修正话术不会改变 Router 的错误工具选择。')  # 解释末端修复无效。

最后错误基线: E5
基线风险：修正话术不会改变 Router 的错误工具选择。


In [3]:
by_id = {row['id']:row for row in events}  # 建立事件 ID 到节点的索引。
def first_error(node_id):  # 定义沿 parent 边回溯的首错定位函数。
    chain = []  # 初始化从当前错误到根的因果链。
    current = by_id[node_id]  # 从最终坏节点开始向上追溯。
    while current is not None:  # 在仍有父事件时继续回溯。
        chain.append(current)  # 保存当前节点到因果链。
        current = by_id.get(current['parent'])  # 读取父节点或到达根。
    broken = [row for row in reversed(chain) if not row['ok']]  # 从根到叶找第一个实际违规节点。
    return broken[0]['id'], [row['id'] for row in reversed(chain)]  # 返回首错 ID 与完整传播路径。

In [4]:
root, chain = first_error('E5')  # 从虚假完成回复回溯其因果错误。
print('最终错误:', last_bad)  # 输出末端错误节点。
print('首个因果错误:', root)  # 输出真正应修复的 Router 节点。
print('传播链:', chain)  # 输出从目标到错误回答的完整依赖路径。
print('结果解读：E3-E5 是 E2 产生错误 artifact 后的传播，而 E6 是独立正确分支。')  # 解释因果分层。

最终错误: E5
首个因果错误: E2
传播链: ['E1', 'E2', 'E3', 'E4', 'E5']
结果解读：E3-E5 是 E2 产生错误 artifact 后的传播，而 E6 是独立正确分支。


In [5]:
wrong = last_bad  # 保留只修最后回答的错误定位。
fixed = root  # 读取依赖回溯得到的首错。
print('失败案例：末端归因=', wrong, '，首错归因=', fixed)  # 展示修复位置的差异。
print('生产差距：需记录 artifact hash、输入输出版本、并行因果边与反事实 fixture，才能区分相关和根因。')  # 说明因果评分需要的数据。

失败案例：末端归因= E5 ，首错归因= E2
生产差距：需记录 artifact hash、输入输出版本、并行因果边与反事实 fixture，才能区分相关和根因。


In [6]:
assert root == 'E2'  # 验证 Router 选错是该传播链的首个因果错误。
assert last_bad == 'E5'  # 验证末端错误与根因不同。
assert 'E6' not in chain  # 验证独立正确分支不会被错误归因。